# Generators

De ce ? 

-   Cand lucram cu situatii in care avem multe date si vrem sa evitam crash uri din cauza memoriei insuficiente

Ce face un Generator ?

-   Pune pauza la functie, returneaza o valoare folosind yield, apoi continua executia functiei. Deci dam pauza apoi continue daca vrem din programul principal

Cu ce ne ajuta ?

-   Memory Use: Efficient

-   Value Generation: 1 at a time

-   Modularity: Yes

-   Readability

-   Maintainability

E modular pentru ca de exemplu cand generam niste numere dupa o regula, logica de generare poate fi separata de logica de filtrare ulterioara.

In [1]:
def my_gen():
    # Code
    value = 1
    yield value
    # Code

# Main Program

In [2]:
def gen_values():
    yield 1
    yield 2
    yield 3
    
gen_values()

<generator object gen_values at 0x739c78ef5430>

### Obs

-   Primim ca rezultat un obiect de tip generator pentru ca executia este de tip lazy (memory efficient) si nu executa primul yield decat daca ii ceri sa o faca

-   Spre deosebire de o functie clasica cu return, care este de tip eager si executa imediat codul

In [3]:
def gen_values():
    yield 1
    yield 2
    yield 3
    
gen_values_obj = gen_values()
print(next(gen_values_obj))
print(next(gen_values_obj))

1
2


Cu next() ii spunem sa execute efectiv yield ul curent, dar e ineficient sa scrii 5 next() uri. Asa ca se poate folosi un for in schimb, si pentru a evita eroarea in care sari de ultimul yield.

In [4]:
def gen_values():
    yield 1
    yield 2
    yield 3
    
gen_values_obj = gen_values()
for value in gen_values_obj:
    print(value)

1
2
3


In [5]:
def get_prime_list(start, end):
    primes = []
    for num in range(start, end + 1):
        if num < 2:
            continue
        is_prime = True
        for i in range(2, num):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(num)
    
    return primes

print(get_prime_list(50, 100))

[53, 59, 61, 67, 71, 73, 79, 83, 89, 97]


In [9]:
def get_prime_list(start, end):
    for num in range(start, end + 1):
        if num < 2:
            continue
        is_prime = True
        for i in range(2, num):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            yield num

for prime in get_prime_list(50, 100):
    print(prime)

53
59
61
67
71
73
79
83
89
97


### Ca minus

-   Nu ne putem intoarce la o valoare anterioara. Ce a fost s-a dus.

-   Ca solutie -> Salvam obiectul sub forma de lista

In [10]:
def get_prime_list(start, end):
    for num in range(start, end + 1):
        if num < 2:
            continue
        is_prime = True
        for i in range(2, num):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            yield num

object_prime = list(get_prime_list(50, 100))

for prime in object_prime:
    print(prime)

53
59
61
67
71
73
79
83
89
97


Daca de exemplu vrem sa citim un fisier super mare:

In [13]:
def read_file(file_path):
    with open(file_path) as file:
        for line in file:
            yield line.strip()

Va fi eficient pe partea de memorie pentru ca nu va incarca toate acele informatii din fisier, ci in cazul asta va intoarce cate o linie pe rand. 

# Iterator

-   Un obiect care returneaza elemente "1 at a time" from a sequence (or data stream) and remembers its position between calls

In [14]:
import random

class Dice:
    def __init__(self, rolls):
        self.rolls = rolls
        self.count = 0
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.count < self.rolls:
            self.count += 1
            return random.randint(1, 6)
        else:
            raise StopIteration

In [17]:
dice = Dice(3)

for die in dice:
    print(die)

2
4
1


In [19]:
dice = [die for die in Dice(3)]
dice

[3, 4, 3]

### Generator Expressions (Varianta "One-Liner")

-   Exista o diferenta mare de performanta

In [20]:
# List comprehension - Încarcă totul în memorie IMEDIAT
liste_patrate = [x**2 for x in range(1000000)] 

# Generator expression - NU ocupă memorie până nu ceri valorile
gen_patrate = (x**2 for x in range(1000000))

-   Cum creezi un obiect iterabil pentru un miliard de elemente fără să ocupi 8GB de RAM?

Easy -> Generator expression pentru ca nu ocupa memorie pana nu cer elementele

### Diferenta dintre Iterable si Iterator (Concept Teoretic Clar)

-   Iterable -> Un obiect pe care poti apela iter() si care iti da un iterator (ex: list, dict, str). Are metoda __iter__

-   Iterator -> Obiectul care retine starea si produce urmatoarea valoarea prin __next__. Un iterator este mereu un Iterable, dar un Iterable (ex lista) nu este mereu un Iterator.

### Pipeline-uri de Generator (Data Engineering Pattern)

-   Generatorii sunt perfecti pentru inlantuirea de operatiuni, deoarece datelel 'curg' prin pipelines fara a fi stocate intermediar

In [ ]:
def read_logs(file):
    for line in file:
        yield line
        
def filter_errors(lines):
    for line in lines:
        if "ERROR" in line:
            yield line
            
def format_msg(errors):
    for error in errors:
        yield error.upper()
        
raw_data = read_logs(open("big_data.log"))
errors = filter_errors(raw_data)
final_output = format_msg(errors)

### Libraria itertools (Power User Tools)

-   islice: Pentru a lua doar o portiune dintr-un generator (ca un slice pe lista, dar fara a consuma tot generatorul)

-   chain: Pentru a lipi mai multi iteratori fara a crea liste noi

-   cycle: Pentru a repeta un set de date la infinit



### yield from

-   Modalitate mai eleganta de a returna valori dintr-un generator subordonat

-   Pentru cod mai curat in pipeline-uri complexe

In [22]:
def sub_gen():
    yield 1
    yield 2
    
def main_gen():
    yield from sub_gen()
    yield 3

### Consumarea Generatorului

-   Un generator poate fi parcurs o singura data

Ce se intampla daca fac sum(my_gen) si apoi max(my_gen) ?

-   Easy -> sum va functiona, dar max va da eroare pentru ca generatorul a fost deja epuizat cand am facut sum ul